# 31 · Which PT pathway analysis should the paper use?

**Goal.** Pick one pathway analysis for the human-versus-mouse PT comparison and justify the
choice. Then use it to find the pathways where pseudospace tells us something that
conventional analysis cannot.

**Inputs.** The PT structures, labels and nonbranching scFates coordinate come from notebook 13.
The gene universe, pathway libraries and planned sensitivity runs come from notebook 12. This
notebook refits nothing upstream. It adds one control that notebook 12 lacks: **balanced specimen
relabeling**.

**Cohort.** Two control mice and two cortex sections from one human donor. The section labelled
"medulla" is also healthy cortex. All four specimens are male (each expresses Y-linked genes), so
sex does not separate the species. Every result is a descriptive within-cohort comparison.

| Part | Question |
|---|---|
| A · Method selection | Which strategies report pathways for the species contrast but not under relabeled specimens? |
| B · Final pathway list | Which pathways remain after specificity, stability and correlation checks? |
| C · What pseudospace adds | Which pathways do conventional analyses miss, and what do positions and genes reveal? |

## 1 · Folders and fixed settings

Set `PSEUDOSPACE_DATA_ROOT` and `PSEUDOSPACE_RESULTS_ROOT`, or pass `--data-root` and
`--results-root`. All thresholds below are fixed before any result in this notebook is read.

In [ ]:
import argparse
import json
import os
import sys
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
upstream13 = results_root / 'minimal_pt_scfates'
upstream12 = results_root / 'pt_pathway_remodeling_scfates'
output = results_root / 'pt_pathway_method_selection'
figures = output / 'figures'
figures.mkdir(parents=True, exist_ok=True)
required = [upstream13 / 'cross_species_pt_scfates.h5ad', upstream13 / 'ortholog_map_used.csv',
            upstream12 / 'gene_statistics.csv', upstream12 / 'pathway_evidence_atlas.csv']
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f'Run notebooks 13 and 12 first: missing {path}')

NOTEBOOK_LOGIC_VERSION = '31.method_selection.1'  # Bump when a cached calculation changes.
ALPHA = .05          # BH level for every screen.
N_NULL = 9999        # Matched random sets per pathway, as in notebook 12.
SEED = 12
BASIS_DF = 6         # Spline basis of notebook 12.
N_BINS = 6           # Equal-width step model, nearly matching the spline's 6 added columns.
MIN_RETENTION = 5 / 7  # Retained in at least 5 of notebook 12's 7 planned sensitivity runs.
print('Results folder:', output)

## 2 · Rebuild notebook 12's exact inputs

The same PT structures, common coordinate support, count-based normalization, 2% equal-specimen
detection rule and 10–300-gene pathway rule as notebook 12. The guard at the end stops if the
gene universe or pathway family differs from notebook 12's saved run.

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display
from pseudospace.pathway_inputs import rebuild_pt_expression
from pseudospace.pathway_remodeling import gene_covariates, matching_strata
from pseudospace.pathways import build_pathway_membership

saved_pt = ad.read_h5ad(upstream13 / 'cross_species_pt_scfates.h5ad', backed='r')
pt_obs = saved_pt.obs.copy()
saved_pt.file.close()
pt_obs = pt_obs[pt_obs.broad_tubule_marker_call.astype(str).eq('PT')].copy()
expected = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
actual = pt_obs.groupby('comparison_species', observed=True)['sample'].apply(
    lambda v: sorted(v.astype(str).unique())).to_dict()
if actual != expected:
    raise ValueError(f'Unexpected cohort: {actual}')
support = pt_obs.groupby('sample', observed=True).shared_pseudospace.quantile([.01, .99]).unstack()
coordinate = pt_obs.shared_pseudospace.to_numpy(float)
pt_obs = pt_obs[(coordinate >= support[.01].max()) & (coordinate <= support[.99].min())].copy()
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
source_paths = {name: data_root / 'tubule_by_gene' / f'{name}_tubule_by_gene_caleb.h5ad'
                for names in expected.values() for name in names}
adata = rebuild_pt_expression(pt_obs, source_paths, orthologs, target_sum=1e4)
position = adata.obs.shared_pseudospace.to_numpy(float)
human = adata.obs.comparison_species.astype(str).eq('human').to_numpy()
specimen = adata.obs['sample'].astype(str).to_numpy()
segment = adata.obs.segment_class.astype(str).to_numpy()
lo = max(position[specimen == name].min() for name in expected['mouse'] + expected['human'])
hi = min(position[specimen == name].max() for name in expected['mouse'] + expected['human'])
grid = np.linspace(lo, hi, 61)

expression = sparse.csr_matrix(adata.layers['lognorm'], dtype=float)
universe = gene_covariates(expression, position, human, specimen, adata.var_names)
universe['measured_in_both'] = adata.var.measured_in_both_inputs.to_numpy()
universe['eligible'] = universe.measured_in_both & universe.detection.ge(.02)
genes = universe.loc[universe.eligible, 'gene'].tolist()
Y = expression[:, universe.eligible.to_numpy()]
counts = sparse.csr_matrix(adata.layers['counts'], dtype=float)[:, universe.eligible.to_numpy()]
covariates = universe.set_index('gene').loc[genes]
strata = matching_strata(covariates, bins=3)

libraries = ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')
coverage = pd.concat([build_pathway_membership(
    json.loads((data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{library}.json').read_text()),
    universe.loc[universe.measured_in_both, 'gene'], ortholog_map=orthologs, library_name=library,
    tested=genes, min_genes=1) for library in libraries], ignore_index=True)
coverage['pathway_id'] = coverage.library + '::' + coverage.pathway
coverage = coverage[coverage.n_tested.between(10, min(300, len(genes) - 1))]
gene_sets = {row.pathway_id: list(row.genes_present) for row in coverage.itertuples()}

saved_gene_stats = pd.read_csv(upstream12 / 'gene_statistics.csv', index_col=0)
atlas12 = pd.read_csv(upstream12 / 'pathway_evidence_atlas.csv')
if list(saved_gene_stats.index) != genes or set(atlas12.pathway_id) != set(gene_sets):
    raise ValueError('Gene universe or pathway family differs from notebook 12; rerun notebook 12.')
display(adata.obs.groupby(['comparison_species', 'sample', 'segment_class'], observed=True).size()
        .unstack().rename_axis(columns=None))
print(f'{len(position):,} structures · {len(genes):,} genes · {len(gene_sets):,} pathways · '
      f'common interval {lo:.3f}–{hi:.3f}')

## 3 · The candidate strategies and how we choose

Every strategy below uses the same structures, coordinate, genes and pathway sets.

| # | Strategy | Gene or pathway statistic | Pathway test | Asks |
|---|---|---|---|---|
| 1 | Whole-PT pseudobulk (conventional) | DESeq2 Wald on specimen sums | weighted preranked GSEA | Is the pathway shifted on average? |
| 2 | S1/S2/S3 pseudobulk (conventional) | DESeq2 Wald within each reviewed segment | weighted preranked GSEA, any segment | Is it shifted in a coarse region? |
| 3 | Pathway score along pseudospace | module score per structure → nested GAM F-test | parametric F, structures as replicates | Does the pathway score differ by position? |
| 4 | Gene GAM, constant offset `T_level` | species offset given a shared curve | covariate-matched rank-AUC | Do member genes differ more than others, on average? |
| 5 | Gene GAM, `T_total` | offset or shape | matched rank-AUC | ...anywhere? |
| 6 | Gene GAM, S1/S2/S3 steps `T_discrete_spatial` | species × reviewed-segment steps beyond the offset | matched rank-AUC | ...differently between segments? |
| 7 | Gene GAM, equal-width steps | species × 6 coordinate bins beyond the offset | matched rank-AUC | ...differently between coordinate bins? |
| 8 | **Gene GAM, smooth `T_spatial`** | species × smooth position beyond the offset | matched rank-AUC | ...differently along continuous position? |

Strategy 8 is also run with two other pathway tests (a size-only random-set null and preranked
GSEA), so the gene statistic and the set test can be judged separately.

**Criteria, in priority order:**

1. **Specificity.** Relabel the four specimens into the two balanced mouse+human groupings and
   rerun every strategy unchanged. These groupings contain no species difference: any pathway a
   strategy reports for them reflects specimen heterogeneity or structure-level pseudoreplication.
   Lamian used the same random-partition control
   ([Hou et al. 2023](https://pmc.ncbi.nlm.nih.gov/articles/PMC10638410)). We summarize this as
   the **negative-control discovery ratio** (NCDR): mean relabeled discoveries ÷ species
   discoveries. We call a strategy specific when NCDR < 0.25 **and** the relabeled groupings
   average fewer calls than 5% of the tested pathways. The second condition was added after the
   first run: a pathway-score model called almost every pathway for species and about 280 for
   relabeled groups, which passes the ratio but is clearly not specific. It changes no
   gene-statistic decision. With only two relabelings, this screens for gross nonspecificity; it
   is not a calibrated false-discovery rate.
2. **Positional information.** The answer should say where along the PT the species differ.
3. **Robustness** under notebook 12's planned leave-one-specimen-out, basis and detection runs.
4. **Simplicity.** Prefer the simpler strategy when the others tie.

## 4 · Gene statistics for the species split and both relabelings

A relabeled group holds one mouse and one human specimen. The species' own smooth shape enters
every relabeled model as a nuisance term. The tested term is then only a difference between
specimens of the same species. Specimen intercepts enter as within-group sum-to-zero contrasts,
exactly as in notebook 12, and every specimen keeps a quarter of the weight. The species-split
statistics must reproduce notebook 12's saved `T_level`, `T_spatial` and `T_total`.

We also compare the two mice with each other, and the two human sections with each other, using
the same models with one specimen per group.

In [ ]:
import inspect
from pseudospace import pathway_calibration as calibration
from pseudospace.pathway_calibration import (balanced_partitions, contrast_designs, nested_partial_f,
                                             normal_matched_q, pathway_scores, parametric_calls,
                                             negative_control_summary)
from pseudospace.pathway_remodeling import fit_nested_trajectories, matched_pathway_tests
import pseudospace.pathway_remodeling as remodeling
from pseudospace.stats_gam import gam_internal_knots
from pseudospace.stage_cache import cached_payload, cached_frame, digest

cache = output / 'stage_cache'
code = digest([NOTEBOOK_LOGIC_VERSION, Path(calibration.__file__), Path(remodeling.__file__)])
inputs = {'Y': digest(Y), 'position': position, 'human': human, 'specimen': specimen,
          'segment': segment, 'genes': genes}
knots = gam_internal_knots(position, basis_df=BASIS_DF)
partitions = balanced_partitions(specimen, np.where(human, 'human', 'mouse'))
bins = np.clip(((position - lo) / (hi - lo) * N_BINS).astype(int), 0, N_BINS - 1).astype(str)
print('Partitions:', partitions)


def partition_scores(label):
    """All gene statistics for one partition as arrays; relabelings absorb the species shape."""
    if label == 'species':
        group, nuisance = human.astype(float), None
    else:
        group, nuisance = np.isin(specimen, partitions[label]).astype(float), human.astype(float)
    designs, weights = contrast_designs(position, group, specimen, knots, segments=segment,
                                        nuisance_shape=nuisance)
    scores = nested_partial_f(Y, designs, weights)
    designs, weights = contrast_designs(position, group, specimen, knots, segments=bins,
                                        nuisance_shape=nuisance)
    binned = nested_partial_f(Y, designs, weights, {'T_bins_spatial': ('level', 'discrete_shape')})
    df = {**scores.pop('df'), **binned.pop('df')}
    return {**scores, **binned, **{'df_' + k: np.array(v) for k, v in df.items()}}


def within_species_scores(pair):
    keep = np.isin(specimen, pair)
    designs, weights = contrast_designs(position[keep], (specimen[keep] == pair[0]).astype(float),
                                        specimen[keep], gam_internal_knots(position[keep], basis_df=BASIS_DF))
    scores = nested_partial_f(Y[keep], designs, weights,
                              {'T_level': ('base', 'level'), 'T_spatial': ('level', 'full')})
    return {k: v for k, v in scores.items() if k != 'df'}


def stage_name(prefix, label):
    return prefix + label.replace(':', '_').replace('+', '_')


gene_scores, design_df = {}, {}
for label in partitions:
    payload = cached_payload(stage_name('gene_scores_', label), lambda label=label: partition_scores(label),
        root=cache, params={'basis_df': BASIS_DF, 'n_bins': N_BINS, 'label': label}, inputs=inputs, code=code)
    gene_scores[label] = pd.DataFrame({k: v for k, v in payload.items() if not k.startswith('df_')}, index=genes)
    design_df[label] = {k[3:]: tuple(int(x) for x in v) for k, v in payload.items() if k.startswith('df_')}
for species_name, pair in expected.items():
    payload = cached_payload('gene_scores_within_' + species_name, lambda pair=pair: within_species_scores(pair),
        root=cache, params={'basis_df': BASIS_DF, 'pair': pair}, inputs=inputs, code=code)
    gene_scores['within_' + species_name] = pd.DataFrame(payload, index=genes)
# Guard, every run: the species split is notebook 12's model.
for name in ('T_level', 'T_spatial', 'T_total'):
    np.testing.assert_allclose(gene_scores['species'][name], saved_gene_stats[name], rtol=1e-6, atol=1e-8)
pd.DataFrame(design_df).T.to_csv(output / 'design_degrees_of_freedom.csv')
median_scores = pd.DataFrame({k: v.median() for k, v in gene_scores.items()}).T
display(median_scores.round(2).rename_axis('median gene score'))

The table already shows the core contrast. For an average gene, the species offset `T_level`
is about 100 times larger than any between-specimen offset, and the smooth species shape
`T_spatial` about 4 times larger. Pathway tests do not use these magnitudes, however. A
competitive test asks only whether a pathway's genes **rank** above other genes. Coherent
specimen differences can therefore produce pathway "discoveries" for any grouping.

## 5 · Run every strategy on every partition

Matched rank-AUC tests use notebook 12's covariate strata and 9,999 random sets. An empirical
p-value cannot fall below 1/10,000, so we also report a normal approximation from the same
matched null. It removes that floor and is used for the specificity counts. The conventional
pipelines repeat notebook 12's DESeq2 and multilevel signed GSEA. A relabeled design adds species
as a covariate (`~species + group`). The pathway-score strategy uses a structure-level F-test,
as is common when pathway scores are fitted along pseudotime.

In [ ]:
from statsmodels.stats.multitest import multipletests

matched_parts = []
for label, frame in gene_scores.items():
    columns = (['T_level', 'T_spatial'] if label.startswith('within_') else
               ['T_level', 'T_spatial', 'T_total', 'T_discrete_spatial', 'T_bins_spatial',
                'T_position_given_segments'])
    result = cached_frame(stage_name('matched_', label),
        lambda frame=frame, columns=columns: matched_pathway_tests(frame[columns], gene_sets, strata,
                                                                   n_null=N_NULL, seed=SEED),
        root=cache, params={'n_null': N_NULL, 'seed': SEED, 'columns': columns},
        inputs={'scores': frame[columns], 'sets': gene_sets, 'strata': strata}, code=code)
    matched_parts.append(result.assign(partition=label))
uniform_parts = []
for label in partitions:
    frame = gene_scores[label][['T_spatial']]
    result = cached_frame(stage_name('uniform_', label),
        lambda frame=frame: matched_pathway_tests(frame, gene_sets, np.zeros(len(genes), int),
                                                  n_null=N_NULL, seed=SEED),
        root=cache, params={'n_null': N_NULL, 'seed': SEED},
        inputs={'scores': frame, 'sets': gene_sets}, code=code)
    uniform_parts.append(result.assign(partition=label))
matched = normal_matched_q(pd.concat(matched_parts, ignore_index=True))
uniform = normal_matched_q(pd.concat(uniform_parts, ignore_index=True))
matched.to_csv(output / 'matched_pathway_tests_all_partitions.csv', index=False)
print(f'{len(matched):,} matched pathway tests across {matched.partition.nunique()} partitions')

In [ ]:
import gseapy as gp
from importlib.metadata import version
from pseudospace.conventional_pt import deseq2_contrast

segments = ('PT-S1', 'PT-S2', 'PT-S3')
names = expected['mouse'] + expected['human']
blocks = {label: pd.DataFrame(np.vstack([np.asarray(counts[(specimen == n) & (segment == label)]
                                                     .sum(axis=0)).ravel() for n in names]),
                               index=names, columns=genes) for label in segments}
count_support = np.logical_and.reduce([blocks[s].ge(10).sum(axis=0).ge(2).to_numpy() for s in segments])
count_genes = pd.Index(genes)[count_support]
count_sets = {p: [g for g in v if g in set(count_genes)] for p, v in gene_sets.items()}
count_sets = {p: v for p, v in count_sets.items() if 10 <= len(v) <= 300}
whole_counts = sum(blocks[s] for s in segments)[count_genes].astype(np.int64)
conventional_code = digest([code, 'conventional-v1', version('pydeseq2'), version('gseapy')])


def signed_gsea(stat):
    ranking = stat.dropna().rename_axis('gene').reset_index(name='stat').sort_values(
        ['stat', 'gene'], ascending=[False, True], kind='stable')
    table = gp.prerank(rnk=ranking, gene_sets=count_sets, min_size=10, max_size=300, weight=1,
                       method='multilevel', seed=SEED, threads=4, outdir=None, no_plot=True).res2d
    table = table.rename(columns={'Term': 'pathway_id'})
    return pd.DataFrame({'pathway_id': table.pathway_id, 'NES': pd.to_numeric(table.NES),
                         'p': pd.to_numeric(table['NOM p-val'])})


def conventional_metadata(label):
    metadata = pd.DataFrame({'species': ['human' if n in expected['human'] else 'mouse' for n in names]},
                            index=names)
    if label == 'species':
        return metadata, '~species', ('species', 'human', 'mouse')
    metadata['group'] = ['a' if n in partitions[label] else 'b' for n in names]
    return metadata, '~species + group', ('group', 'a', 'b')


def conventional_deseq2(label):
    """Whole-PT and per-segment DESeq2 for one partition, on the same count universe."""
    metadata, design, contrast = conventional_metadata(label)
    scopes = [('whole_PT', whole_counts)] + [(s, blocks[s][count_genes].astype(np.int64)) for s in segments]
    return pd.concat([deseq2_contrast(block, metadata, design=design, contrast=contrast).assign(scope=scope)
                      for scope, block in scopes], ignore_index=True)


count_inputs = {'counts': [whole_counts] + [blocks[s][count_genes] for s in segments], 'sets': count_sets}
conventional_de, conventional_gsea = [], []
for label in partitions:
    de = cached_frame(stage_name('deseq2_', label), lambda label=label: conventional_deseq2(label), root=cache,
                      params={'label': label}, inputs=count_inputs, code=conventional_code)
    gsea = cached_frame(stage_name('signed_gsea_', label),
        lambda de=de: pd.concat([signed_gsea(rows.set_index('gene').stat).assign(scope=scope)
                                 for scope, rows in de.groupby('scope', sort=True)], ignore_index=True),
        root=cache, params={'weight': 1, 'method': 'multilevel', 'seed': SEED},
        inputs={'de': de[['gene', 'stat', 'scope']], 'sets': count_sets}, code=conventional_code)
    conventional_de.append(de.assign(partition=label))
    conventional_gsea.append(gsea.assign(partition=label))
conventional_de = pd.concat(conventional_de, ignore_index=True)
conventional_gsea = pd.concat(conventional_gsea, ignore_index=True)
# BH within each partition: whole-PT over pathways; segments over pathway x segment, as in notebook 12.
conventional_gsea['family'] = np.where(conventional_gsea.scope.eq('whole_PT'), 'whole_PT', 'segments')
conventional_gsea['q'] = conventional_gsea.groupby(['partition', 'family']).p.transform(
    lambda p: multipletests(p, method='fdr_bh')[1])
conventional_de.to_csv(output / 'conventional_deseq2_all_partitions.csv', index=False)
conventional_gsea.to_csv(output / 'conventional_signed_gsea_all_partitions.csv', index=False)
display(conventional_de.assign(deg=lambda d: d.padj.le(ALPHA)).groupby(['partition', 'scope']).deg.sum()
        .unstack().rename_axis(columns='DESeq2 genes, padj ≤ 0.05'))


def unsigned_gsea(frame):
    ranking = frame.rename_axis('gene').reset_index(name='stat').sort_values(
        ['stat', 'gene'], ascending=[False, True], kind='stable')
    table = gp.prerank(rnk=ranking, gene_sets=gene_sets, min_size=10, max_size=300, weight=0,
                       method='multilevel', seed=SEED, threads=4, outdir=None, no_plot=True).res2d
    table = table.rename(columns={'Term': 'pathway_id'})
    out = pd.DataFrame({'pathway_id': table.pathway_id, 'NES': pd.to_numeric(table.NES),
                        'p': pd.to_numeric(table['NOM p-val'])})
    out['q'] = multipletests(out.p, method='fdr_bh')[1]
    return out


spatial_gsea = pd.concat([cached_frame(stage_name('spatial_gsea_', label),
    lambda label=label: unsigned_gsea(gene_scores[label].T_spatial), root=cache,
    params={'weight': 0, 'method': 'multilevel', 'seed': SEED},
    inputs={'scores': gene_scores[label].T_spatial, 'sets': gene_sets}, code=conventional_code)
    .assign(partition=label) for label in partitions], ignore_index=True)

score_parts = []
for label in partitions:
    group = human.astype(float) if label == 'species' else np.isin(specimen, partitions[label]).astype(float)
    designs, weights = contrast_designs(position, group, specimen, knots, segments=segment,
                                        nuisance_shape=None if label == 'species' else human.astype(float))
    module_scores = pathway_scores(Y, gene_sets, genes, weights)
    result = nested_partial_f(module_scores.to_numpy(), designs, weights,
                              {'T_level': ('base', 'level'), 'T_spatial': ('level', 'full')})
    df = result.pop('df')
    result = {k: pd.Series(v, index=module_scores.columns) for k, v in result.items()}
    score_parts.append(parametric_calls(result, df, ALPHA).assign(partition=label))
score_calls = pd.concat(score_parts, ignore_index=True)

## 6 · Figure 1 · Which strategies are specific?

A specific strategy reports pathways for the species split (filled marker) and few or none for
the two relabeled groupings (open gray markers). The right margin gives the NCDR and the
relabeled calls as a share of tested pathways. Matched-test calls use positive enrichment and the
normal-approximation BH q ≤ 0.05. GSEA and F-test calls use their own BH q ≤ 0.05, in either NES
direction for signed GSEA.

In [ ]:
hit_rows = []
strategy_labels = {
    'T_level': '4 · Gene GAM offset (T_level)', 'T_total': '5 · Gene GAM total (T_total)',
    'T_discrete_spatial': '6 · Gene GAM × S1/S2/S3 steps', 'T_bins_spatial': f'7 · Gene GAM × {N_BINS} equal bins',
    'T_spatial': '8 · Gene GAM × smooth position (T_spatial)',
    'T_position_given_segments': 'Gene GAM: smooth position beyond S1/S2/S3 steps'}
for stat, label in strategy_labels.items():
    rows = matched[matched.statistic.eq(stat) & matched.partition.isin(list(partitions))]
    hit_rows.append(pd.DataFrame({'strategy': label, 'partition': rows.partition, 'pathway_id': rows.pathway_id,
                                  'hit': rows.effect.gt(0) & rows.q_normal.le(ALPHA)}))
rows = uniform[uniform.statistic.eq('T_spatial')]
hit_rows.append(pd.DataFrame({'strategy': '8b · T_spatial, size-only random sets', 'partition': rows.partition,
                              'pathway_id': rows.pathway_id, 'hit': rows.effect.gt(0) & rows.q_normal.le(ALPHA)}))
hit_rows.append(pd.DataFrame({'strategy': '8c · T_spatial, preranked GSEA', 'partition': spatial_gsea.partition,
                              'pathway_id': spatial_gsea.pathway_id,
                              'hit': spatial_gsea.NES.gt(0) & spatial_gsea.q.le(ALPHA)}))
whole = conventional_gsea[conventional_gsea.scope.eq('whole_PT')]
hit_rows.append(pd.DataFrame({'strategy': '1 · Whole-PT pseudobulk + GSEA', 'partition': whole.partition,
                              'pathway_id': whole.pathway_id, 'hit': whole.q.le(ALPHA)}))
seg_any = (conventional_gsea[conventional_gsea.family.eq('segments')]
           .groupby(['partition', 'pathway_id']).q.min().reset_index())
hit_rows.append(pd.DataFrame({'strategy': '2 · S1/S2/S3 pseudobulk + GSEA', 'partition': seg_any.partition,
                              'pathway_id': seg_any.pathway_id, 'hit': seg_any.q.le(ALPHA)}))
for stat, label in (('T_level', '3a · Pathway score GAM, offset'), ('T_spatial', '3b · Pathway score GAM, shape')):
    rows = score_calls[score_calls.statistic.eq(stat)]
    hit_rows.append(pd.DataFrame({'strategy': label, 'partition': rows.partition,
                                  'pathway_id': rows.pathway_id, 'hit': rows.hit}))
hits = pd.concat(hit_rows, ignore_index=True)
hits.to_csv(output / 'strategy_hits_all_partitions.csv', index=False)
specificity = negative_control_summary(hits)
specificity['tested'] = hits[hits.partition.eq('species')].groupby('strategy').size()
specificity['null_rate'] = specificity.null_mean / specificity.tested
specificity['specific'] = specificity.ncdr.lt(.25) & specificity.null_rate.lt(.05)
order = sorted(specificity.index, key=lambda s: (s[0].isdigit() is False, s))
specificity = specificity.loc[order]
specificity.to_csv(output / 'strategy_specificity.csv')
display(specificity.round(2))

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 9,
                     'axes.titlesize': 10, 'axes.titleweight': 'bold', 'svg.fonttype': 'none',
                     'pdf.fonttype': 42})
HUMAN, MOUSE, NULL, INK = '#D55E00', '#0072B2', '#9A9A9A', '#333333'


def save(fig, name):
    for suffix in ('pdf', 'png'):
        fig.savefig(figures / f'{name}.{suffix}', bbox_inches='tight', dpi=200)


null_columns = [c for c in specificity if c.startswith('hits_swap')]
groups = [('Conventional pseudobulk', ('1 ·', '2 ·')), ('Pathway score along pseudospace', ('3a', '3b')),
          ('Gene GAM: offset', ('4 ·', '5 ·')), ('Gene GAM: position', ('6 ·', '7 ·', '8 ·')),
          ('Other set tests on T_spatial', ('8b', '8c')), ('Secondary', ('Gene GAM:',))]
fig, ax = plt.subplots(figsize=(7.8, 5.6), layout='constrained')
y = np.arange(len(specificity))[::-1].astype(float)
for yy, (name, row) in zip(y, specificity.iterrows()):
    values = [row.true_hits] + [row[c] for c in null_columns]
    ax.plot([min(values), max(values)], [yy, yy], color='#DDDDDD', lw=1.2, zorder=1)
    for c, marker in zip(null_columns, ('s', 'D')):
        ax.scatter(row[c], yy, s=32, marker=marker, color='white', edgecolor=NULL, lw=1.3, zorder=2)
    color = HUMAN if row.specific else INK
    ax.scatter(row.true_hits, yy, s=48, color=color, edgecolor='white', lw=1, zorder=3)
    note = f'NCDR {row.ncdr:.2f} · null {row.null_rate:.0%}' if np.isfinite(row.ncdr) else 'NCDR –'
    ax.text(1.02, yy, note, transform=ax.get_yaxis_transform(), va='center', fontsize=7.5,
            color=HUMAN if row.specific else '#555555')
starts = [next(i for i, n in enumerate(specificity.index) if n.startswith(prefixes)) for _, prefixes in groups]
for (title, _), start in zip(groups, starts):
    if start:
        ax.axhline(y[start] + .5, color='#CCCCCC', lw=.6, ls=':')
    ax.text(-0.02, y[start] + .42, title, transform=ax.get_yaxis_transform(), ha='right', va='bottom',
            fontsize=7, color='#777777', style='italic')
ax.set_yticks(y, specificity.index)
ax.set_xscale('symlog', linthresh=10)
ax.set_xlim(left=-0.5)
ax.set_xlabel('Pathways called (BH q ≤ 0.05; symmetric-log scale)')
ax.scatter([], [], s=48, color=INK, label='Species split')
ax.scatter([], [], s=32, marker='s', color='white', edgecolor=NULL, lw=1.3, label='Relabeling 1')
ax.scatter([], [], s=32, marker='D', color='white', edgecolor=NULL, lw=1.3, label='Relabeling 2')
ax.scatter([], [], s=48, color=HUMAN, label='Specific')
ax.legend(frameon=False, loc='lower center', bbox_to_anchor=(.45, 1.0), ncol=4, fontsize=8)
ax.set_title('Only smooth position-dependent screens are specific to the species contrast', loc='left', pad=22)
save(fig, 'fig1_strategy_specificity')
plt.show()
plt.close(fig)

**Reading Figure 1.**

- **Constant-offset screens are not specific.** Whole-PT pseudobulk GSEA, S1/S2/S3 GSEA and the
  gene-GAM offset report as many pathways, or more, for groupings that contain no species
  difference. The gene statistics themselves are well behaved: DESeq2 calls thousands of species
  genes and almost none for relabeled groups (table above). The pathway layer is the problem. A
  gene-permutation enrichment test treats genes as independent, but specimen-level variation
  moves whole pathways together
  ([Goeman & Bühlmann 2007](https://doi.org/10.1093/bioinformatics/btm051);
  [Wu & Smyth 2012](https://doi.org/10.1093/nar/gks461)). So with two specimens per species, an
  average-level pathway difference between species cannot be told apart from specimen variation.
- **Pathway scores tested at the structure level call nearly everything**, relabeled or not.
  Thousands of structures are not thousands of replicates
  ([Squair et al. 2021](https://doi.org/10.1038/s41467-021-25960-2)).
- **Step-shaped position effects are only partly specific**, whether they use reviewed segments
  or equal-width bins.
- **Smooth species-by-position remodeling (`T_spatial`) is the specific screen.** It is the
  pseudospace question: does the species difference change continuously along the PT? Specimen
  offsets cancel in it, as in a difference-in-differences. The set test matters less than the
  gene statistic: the size-only null and GSEA give similar specificity on `T_spatial`.

### Same check within a species

Comparing the two mice, or the two human sections, removes species altogether. Specimen
differences appear as pathway-coherent **offsets**, but hardly as smooth positional remodeling.

In [ ]:
within = matched[matched.partition.str.startswith('within_')].copy()
within['hit'] = within.effect.gt(0) & within.q_normal.le(ALPHA)
within_table = within.pivot_table(index='partition', columns='statistic', values='hit', aggfunc='sum')
within_table.loc['species split'] = [specificity.loc['4 · Gene GAM offset (T_level)', 'true_hits'],
                                     specificity.loc['8 · Gene GAM × smooth position (T_spatial)', 'true_hits']]
within_table = within_table.rename(index={'within_mouse': 'mouse vs mouse', 'within_human': 'human section vs section'})
within_table.to_csv(output / 'within_species_controls.csv')
display(within_table)
fig, ax = plt.subplots(figsize=(5.2, 2.6), layout='constrained')
x = np.arange(len(within_table))
ax.bar(x - .19, within_table.T_level, width=.36, color='#BBBBBB', label='Constant offset (T_level)')
ax.bar(x + .19, within_table.T_spatial, width=.36, color=HUMAN, label='Smooth position (T_spatial)')
for xi, (a, b) in enumerate(within_table[['T_level', 'T_spatial']].to_numpy()):
    ax.text(xi - .19, a, f'{a:.0f}', ha='center', va='bottom', fontsize=8)
    ax.text(xi + .19, b, f'{b:.0f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x, within_table.index)
ax.set_ylabel('Pathways called')
ax.legend(frameon=False, fontsize=8)
ax.set_title('Specimen differences are offsets, not positional remodeling', loc='left')
save(fig, 'fig1b_within_species_controls')
plt.show()
plt.close(fig)

## 7 · Decision

**Primary analysis: gene-level smooth species-by-position statistic (`T_spatial`) from the nested
GAM on the scFates coordinate, tested with notebook 12's covariate-matched rank-AUC.** It keeps
notebook 12's primary list unchanged, adds the relabeling control, and reads positions back from
the local divergence curves D(s) and S(s).

- It is the only strategy that passes the specificity criterion.
- It answers the positional question directly.
- It is simple: one gene model, one competitive test. A self-contained pathway-score model is
  simpler still, but it fails specificity badly.

Constant-offset results (whole-PT, S1/S2/S3, `T_level`) are reported as context only. They can
be real species biology, but this cohort cannot separate them from specimen variation.
Conventional analyses are therefore not "wrong"; they ask a question the design cannot answer
specifically.

## 8 · Final pathway list: a transparent funnel

Start from notebook 12's spatial candidates (positive effect, empirical matched q ≤ 0.05, 9,999
sets) and apply three filters in order:

1. **Not specimen-sensitive**: not called on `T_spatial` (normal-approximation q ≤ 0.05) in
   either relabeled grouping or either within-species comparison.
2. **Stable**: retained in at least 5 of notebook 12's 7 planned runs (each specimen left out,
   basis 4 and 8, 5% detection).
3. **Correlation-supported**: notebook 12's CAMERA-style correlation-adjusted q ≤ 0.05.

In [ ]:
species_spatial = matched[matched.partition.eq('species') & matched.statistic.eq('T_spatial')].set_index('pathway_id')
np.testing.assert_allclose(species_spatial.loc[atlas12.pathway_id, 'auc'].to_numpy(),
                           0.5 + atlas12.effect_T_spatial.to_numpy(), atol=1e-6)
null_spatial = matched[matched.statistic.eq('T_spatial') & ~matched.partition.eq('species')]
sensitive = set(null_spatial.loc[null_spatial.effect.gt(0) & null_spatial.q_normal.le(ALPHA), 'pathway_id'])
final = atlas12.copy()
final['name'] = final.pathway.str.replace(r' R-HSA-\d+$', '', regex=True)
final['candidate'] = final.discovery_candidate.astype('boolean').fillna(False).astype(bool)
final['specimen_sensitive'] = final.pathway_id.isin(sensitive)
final['stable'] = final.retention_fraction.ge(MIN_RETENTION - 1e-9)
final['correlation_supported'] = final.q_corr.le(ALPHA)
final['confident'] = final.candidate & ~final.specimen_sensitive & final.stable & final.correlation_supported
funnel = pd.Series({
    'Spatial candidates (notebook 12)': int(final.candidate.sum()),
    'not specimen-sensitive': int((final.candidate & ~final.specimen_sensitive).sum()),
    'and stable': int((final.candidate & ~final.specimen_sensitive & final.stable).sum()),
    'and correlation-supported': int(final.confident.sum())})
funnel.rename('pathways').to_csv(output / 'final_list_funnel.csv')
display(funnel.to_frame('pathways'))
removed = final[final.candidate & ~final.confident]
display(removed[['name', 'library', 'specimen_sensitive', 'retention_fraction', 'q_corr']]
        .sort_values('retention_fraction').round(3))

## 9 · What the conventional analyses saw for the same pathways

For each confident pathway, check the conventional species calls (whole-PT and any-segment signed
GSEA in the same common count universe) and ask whether each call is also made for a relabeled
grouping.

- **New with pseudospace**: no conventional species call.
- **Conventional call not specific**: a conventional species call that a relabeled grouping
  also produces.
- **Conventional call, localized here**: a conventional species call that no relabeling produces.
  Pseudospace adds where and how the pathway changes.
- **Not testable conventionally**: fewer than 10 members pass the pseudobulk count-support rule.

In [ ]:
conv_hits = hits[hits.strategy.str.startswith(('1 ·', '2 ·')) & hits.hit]
conv_species = set(conv_hits.loc[conv_hits.partition.eq('species'), 'pathway_id'])
conv_null = set(conv_hits.loc[~conv_hits.partition.eq('species'), 'pathway_id'])
final['conventional_species_call'] = final.pathway_id.isin(conv_species)
final['conventional_call_also_relabeled'] = final.pathway_id.isin(conv_species & conv_null)
final['tested_in_count_universe'] = final.pathway_id.isin(count_sets)
final['pseudospace_class'] = np.select(
    [~final.tested_in_count_universe, ~final.conventional_species_call, final.conventional_call_also_relabeled],
    ['not testable conventionally', 'new with pseudospace', 'conventional call not specific'],
    'conventional call, localized here')
final.loc[~final.confident, 'pseudospace_class'] = ''
class_counts = final.loc[final.confident, 'pseudospace_class'].value_counts()
class_counts.rename('confident pathways').to_csv(output / 'pseudospace_class_counts.csv')
display(class_counts.to_frame('confident pathways'))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8.4, 2.6), layout='constrained',
                               gridspec_kw={'width_ratios': [1.2, 1]})
labels_funnel = list(funnel.index)
ax1.barh(np.arange(len(funnel))[::-1], funnel.values, color=['#BBBBBB'] * 3 + [HUMAN], height=.6)
for yy, v in zip(np.arange(len(funnel))[::-1], funnel.values):
    ax1.text(v + 1, yy, str(v), va='center', fontsize=8)
ax1.set_yticks(np.arange(len(funnel))[::-1], labels_funnel)
ax1.set_xlabel('Pathways')
ax1.set_title('Final list', loc='left')
klass = ['new with pseudospace', 'conventional call not specific', 'conventional call, localized here',
         'not testable conventionally']
values = [int(class_counts.get(k, 0)) for k in klass]
ax2.barh(np.arange(4)[::-1], values, color=[HUMAN, '#E8A06A', '#BBBBBB', '#DDDDDD'], height=.6)
for yy, v in zip(np.arange(4)[::-1], values):
    ax2.text(v + 1, yy, str(v), va='center', fontsize=8)
ax2.set_yticks(np.arange(4)[::-1], klass)
ax2.set_xlabel('Confident pathways')
ax2.set_title('Versus conventional analysis', loc='left')
save(fig, 'fig2_final_list_funnel')
plt.show()
plt.close(fig)

## 10 · Group the final pathways into biological themes

Themes are assigned by keyword rules **after** the list is frozen. They are for reading only and
play no part in testing. The first matching rule wins, and every assignment is exported.

In [ ]:
import re

THEMES = [
    ('Xenobiotic & drug metabolism', r'xenobiotic|cytochrome p450|drug metabolism|adme|phase ii|glucuronid|'
                                     r'glutathione|biological oxidations|chemical carcinogenesis|'
                                     r'pentose and glucuronate|ascorbate and aldarate'),
    ('Vitamins & cofactors', r'vitamin|retino|retinol|pantothenate|coa biosynthesis|cofactor|ra biosynthesis'),
    ('Solute transport', r'transport|slc|abc|bicarbonate reclamation|mineral absorption|bile secretion|acid secretion'),
    ('Fatty-acid oxidation & peroxisome', r'fatty acid|peroxisom|ppar|adipogenesis|butanoate|propanoate'),
    ('Amino-acid metabolism', r'amino acid|valine|leucine|tryptophan|lysine|glycine|serine|histidine|cysteine|'
                              r'methionine|alanine|aspartate|glutamate|sulfur|glyoxylate|nitrogen|branched-chain|'
                              r'polyamine|ornithine'),
    ('Glucose & energy metabolism', r'glycoly|gluconeogenesis|pyruvate|citrate cycle|oxidative phosphorylation|glycerolipid'),
    ('Sterol, steroid & bile acid', r'cholesterol|steroid|bile acid|terpenoid|srebp|lipoprotein|sphingolipid|glycerophospholipid'),
]


def theme_of(name):
    return next((theme for theme, rule in THEMES if re.search(rule, name, re.I)), 'Other / signaling')


final['theme'] = final.name.map(theme_of)
confident = final[final.confident].copy()
theme_counts = confident.groupby('theme').agg(pathways=('pathway_id', 'size'),
    new_with_pseudospace=('pseudospace_class', lambda v: int(v.eq('new with pseudospace').sum())),
    median_peak=('peak_position', 'median'),
    human_high_at_peak=('median_member_z_at_peak', lambda v: int(v.gt(0).sum())))
theme_counts.to_csv(output / 'theme_summary.csv')
display(theme_counts.sort_values('pathways', ascending=False))

## 11 · Figure 3 · A positional atlas of species pathway remodeling

Each row is a confident pathway. Color is the **median fitted species difference** of its
member genes at each position: human minus mouse, in log-normalized expression units, from the
full GAM. Red is human-high, blue mouse-high. A black tick marks where the pathway's local
divergence D(s) peaks, that is, where its members separate most from other genes. Rows are
grouped by theme and ordered by that peak. The top strip shows where the reviewed S1/S2/S3
clusters lie on the coordinate (equal-specimen densities). The color scale is clipped at the
98th percentile for display; exported values are unclipped.

In [ ]:
fit = cached_payload('species_nested_fit',
    lambda: {k: v for k, v in fit_nested_trajectories(Y, position, human, specimen, grid,
                                                       basis_df=BASIS_DF).items()},
    root=cache, params={'basis_df': BASIS_DF, 'grid': grid}, inputs=inputs, code=code)
np.testing.assert_allclose(fit['T_spatial'], saved_gene_stats.T_spatial, rtol=1e-6, atol=1e-8)
gene_index = pd.Index(genes)
z = pd.DataFrame(fit['z'], index=genes, columns=np.round(grid, 4))
delta = pd.DataFrame(fit['delta'], index=genes, columns=np.round(grid, 4))
median_delta = pd.DataFrame({p: delta.loc[gene_sets[p]].median() for p in confident.pathway_id}).T
median_delta.to_csv(output / 'confident_pathway_median_delta.csv')

theme_order = [t for t, _ in THEMES] + ['Other / signaling']
confident['theme_rank'] = confident.theme.map({t: i for i, t in enumerate(theme_order)})
confident = confident.sort_values(['theme_rank', 'peak_position'])
rows = median_delta.loc[confident.pathway_id]
fig = plt.figure(figsize=(8.2, 0.15 * len(rows) + 2.2), layout='constrained')
gs = fig.add_gridspec(3, 1, height_ratios=[0.8, 0.15 * len(rows), 0.18])
top = fig.add_subplot(gs[0])
for label, color in zip(segments, ('#BBBBBB', '#888888', '#444444')):
    density = np.zeros(len(grid))
    for name in names:
        values = position[(specimen == name) & (segment == label)]
        density += np.histogram(values, bins=np.r_[grid, grid[-1] + (grid[1] - grid[0])])[0] / (specimen == name).sum()
    top.fill_between(grid, density / 4, color=color, alpha=.5, lw=0)
    top.text(grid[np.argmax(density)], (density / 4).max(), label.replace('PT-', ''), ha='center',
             va='bottom', fontsize=8)
top.set_xlim(grid[0], grid[-1])
top.set(yticks=[], xticks=[])
top.spines['left'].set_visible(False)
top.set_title('Where human and mouse PT pathways diverge along pseudospace', loc='left')
ax = fig.add_subplot(gs[1], sharex=top)
limit = float(np.quantile(np.abs(rows.to_numpy()), .98))
image = ax.imshow(rows.clip(-limit, limit).to_numpy(), aspect='auto', cmap='RdBu_r', vmin=-limit, vmax=limit,
                  extent=(grid[0], grid[-1], len(rows), 0), interpolation='nearest')
ax.scatter(confident.peak_position, np.arange(len(rows)) + .5, marker='|', s=30, color='black', lw=1.2)
ax.set_yticks(np.arange(len(rows)) + .5,
              [('● ' if k == 'new with pseudospace' else '') + n[:46]
               for n, k in zip(confident.name, confident.pseudospace_class)], fontsize=6.3)
boundaries = np.flatnonzero(confident.theme.to_numpy()[1:] != confident.theme.to_numpy()[:-1]) + 1
for b in boundaries:
    ax.axhline(b, color='white', lw=2.5)
starts = np.r_[0, boundaries]
for start, end in zip(starts, np.r_[boundaries, len(rows)]):
    ax.text(1.01, 1 - (start + end) / 2 / len(rows), confident.theme.iloc[start], transform=ax.transAxes,
            ha='left', va='center', fontsize=7.5, fontweight='bold')
ax.set_xlabel('Shared PT pseudospace (scFates)')
cax = fig.add_subplot(gs[2])
fig.colorbar(image, cax=cax, orientation='horizontal',
             label='Median member difference, human − mouse (log-normalized expression)')
save(fig, 'fig3_positional_atlas')
plt.show()
plt.close(fig)
print('● before a name = new with pseudospace (no conventional species call)')

## 12 · Genes that reverse direction along the PT

A whole-PT comparison can see only one direction per gene. Here we count genes whose fitted
species difference has **opposite signs at two positions**, each over a contiguous run of at
least three grid points with |Z| ≥ 3 and an absolute difference of at least 0.25 log-normalized
units. These are display rules for working statistics, not tests. We then ask which confident
pathways contain the most reversing members.

In [ ]:
def reversal_runs(zvalues, dvalues, threshold=3., effect=.25, run=3):
    """True when a curve has >= run contiguous points clearly human-high and clearly mouse-high."""
    def longest(mask):
        best = current = 0
        for v in mask:
            current = current + 1 if v else 0
            best = max(best, current)
        return best
    up = (zvalues >= threshold) & (dvalues >= effect)
    down = (zvalues <= -threshold) & (dvalues <= -effect)
    return longest(up) >= run and longest(down) >= run


reverses = pd.Series([reversal_runs(zr, dr) for zr, dr in zip(fit['z'], fit['delta'])], index=genes,
                     name='reverses')
reverse_genes = reverses[reverses].index
gene_table = saved_gene_stats[['T_level', 'T_spatial', 'Z_level', 'mean_expression', 'detection_mouse',
                               'detection_human']].copy()
gene_table['reverses_along_PT'] = reverses
gene_table['peak_abs_z_position'] = grid[np.abs(fit['z']).argmax(axis=1)]
peak_index = np.abs(fit['delta']).argmax(axis=1)
gene_table['peak_delta_position'] = grid[peak_index]
gene_table['peak_delta'] = fit['delta'][np.arange(len(genes)), peak_index]
gene_table['shape_amplitude'] = fit['delta'].std(axis=1)  # SD of the species difference along PT
gene_table['delta_mean'] = fit['delta'].mean(axis=1)
gene_table['z_early'] = fit['z'][:, :20].mean(axis=1)
gene_table['z_late'] = fit['z'][:, -20:].mean(axis=1)
gene_table.to_csv(output / 'gene_positional_summary.csv')
confident['reversing_members'] = [len(set(gene_sets[p]) & set(reverse_genes)) for p in confident.pathway_id]
confident['reversing_fraction'] = confident.reversing_members / confident.n_tested
print(f'{len(reverse_genes):,} of {len(genes):,} genes reverse species direction along the PT '
      f'({len(reverse_genes) / len(genes):.1%}).')
display(confident.sort_values('reversing_fraction', ascending=False)
        [['name', 'theme', 'n_tested', 'reversing_members', 'reversing_fraction']].head(12).round(2))

## 13 · Theme deep dives: genes, positions and replicate consistency

Each theme gets the same panels.

- **A · Pathways.** Local divergence D(s) of the theme's four strongest confident pathways, with
  the reviewed segments shaded for orientation.
- **B · Member genes.** Fitted human-minus-mouse difference along pseudospace for the theme's
  genes whose species difference **changes most along the PT** (largest SD of the fitted
  difference across positions). Candidates must be detected in at least 10% of structures in one
  species and rank in the top 10% of genes by `T_spatial`. Ranking by `T_spatial` alone would
  favor highly expressed genes with large constant offsets.
- **C · Replicate consistency.** Binned mean expression of four driver genes in **each specimen**.
  Human sections are orange and mice blue; line style separates the two specimens of a species.
  A positional pattern that one specimen alone drives would show here.
- **D · What S1/S2/S3 analysis reports.** DESeq2 human-versus-mouse log2 fold change per
  reviewed segment for the same genes.

The drivers are the two human-high and the two mouse-high genes (at their own peaks) with the
largest positional change, so mixed-direction themes show both sides.

In [ ]:
species_de = conventional_de[conventional_de.partition.eq('species')]
segment_lfc = species_de[species_de.scope.isin(segments)].pivot(index='gene', columns='scope', values='log2FoldChange')
edges = np.linspace(lo, hi, 13)
centres = (edges[:-1] + edges[1:]) / 2
binned_index = np.clip(np.searchsorted(edges, position, side='right') - 1, 0, len(centres) - 1)
spatial_floor = float(gene_table.T_spatial.quantile(.9))
curves = remodeling.local_pathway_curves(fit['z'], genes, {p: gene_sets[p] for p in confident.pathway_id}, grid)
curves.to_csv(output / 'confident_pathway_divergence_curves.csv', index=False)
segment_spans = {s: np.quantile(position[segment == s], [.1, .9]) for s in segments}


def specimen_profiles(gene):
    column = Y[:, gene_index.get_loc(gene)].toarray().ravel()
    return {name: np.array([column[(specimen == name) & (binned_index == b)].mean()
                            if np.any((specimen == name) & (binned_index == b)) else np.nan
                            for b in range(len(centres))]) for name in names}


def theme_genes(theme_rows, n_heat=16):
    members = sorted(set().union(*[gene_sets[p] for p in theme_rows.pathway_id]))
    table = gene_table.loc[members]
    table = table[table[['detection_mouse', 'detection_human']].max(axis=1).ge(.10)
                  & table.T_spatial.ge(spatial_floor)].sort_values('shape_amplitude', ascending=False)
    up = table[table.peak_delta.gt(0)].index[:2].tolist()
    down = table[table.peak_delta.lt(0)].index[:2].tolist()
    drivers = up + down
    drivers += [g for g in table.index if g not in drivers][:4 - len(drivers)]
    return table.head(n_heat), drivers


def short(name, width=34):
    return name if len(name) <= width else name[:width - 1] + '…'


def theme_figure(theme):
    theme_rows = confident[confident.theme.eq(theme)].sort_values(['q_spatial', 'effect_T_spatial'],
                                                                  ascending=[True, False])
    heat, drivers = theme_genes(theme_rows)
    if not drivers:
        print(f'{theme}: no genes pass the display filters; skipped.')
        return []
    fig = plt.figure(figsize=(12, 6.6))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.0, 0.95, 1.3], height_ratios=[1.15, 1], left=.06, right=.985,
                          top=.88, bottom=.12, wspace=.42, hspace=.45)
    ax = fig.add_subplot(gs[0, 0])
    for s, (a, b) in segment_spans.items():
        ax.axvspan(a, b, color='#000000', alpha=.04 + .03 * segments.index(s), lw=0)
        ax.text((a + b) / 2, 1.0, s.replace('PT-', ''), transform=ax.get_xaxis_transform(), ha='center',
                va='bottom', fontsize=7, color='#555555')
    for row, ls in zip(theme_rows.head(4).itertuples(), ('-', '--', '-.', ':')):
        c = curves[curves.pathway_id.eq(row.pathway_id)]
        ax.plot(c.position, c.divergence, color=HUMAN if row.median_member_z_at_peak > 0 else MOUSE,
                ls=ls, lw=1.6, label=short(row.name))
    ax.axhline(0, color='#999999', lw=.7)
    ax.set(xlabel='PT pseudospace', ylabel='Local divergence D(s)')
    handles = ax.get_legend_handles_labels()
    ax.set_title('A · Pathways (orange: human-high at peak)', loc='left', fontsize=9)
    ax = fig.add_subplot(gs[:, 1])
    block = delta.loc[heat.index]
    block = block.loc[block.abs().idxmax(axis=1).sort_values().index]
    limit = max(.5, float(np.quantile(np.abs(block.to_numpy()), .95)))
    im = ax.imshow(block.clip(-limit, limit).to_numpy(), aspect='auto', cmap='RdBu_r', vmin=-limit, vmax=limit,
                   extent=(grid[0], grid[-1], len(block), 0), interpolation='nearest')
    ax.set_yticks(np.arange(len(block)) + .5, [g + (' ◆' if g in drivers else '') for g in block.index], fontsize=7.5)
    ax.set_xlabel('PT pseudospace')
    ax.set_title('B · Genes: human − mouse (◆ drivers)', loc='left', fontsize=9)
    cax = ax.inset_axes([.08, -.12, .84, .022])
    fig.colorbar(im, cax=cax, orientation='horizontal', label='Fitted difference (log-norm)')
    sub = gs[:, 2].subgridspec(len(drivers), 2, width_ratios=[2.3, 1], hspace=.42, wspace=.45)
    records = []
    for i, gene in enumerate(drivers):
        ax = fig.add_subplot(sub[i, 0])
        for name, profile in specimen_profiles(gene).items():
            ax.plot(centres, profile, color=HUMAN if name in expected['human'] else MOUSE,
                    ls='-' if name in (expected['human'][0], expected['mouse'][0]) else '--', lw=1.4)
        ax.text(-.2, .5, gene, transform=ax.transAxes, fontsize=8.5, ha='right', va='center', fontweight='bold')
        ax.tick_params(labelsize=7)
        ax.set_xlim(lo, hi)
        if i < len(drivers) - 1:
            ax.set_xticklabels([])
        else:
            ax.set_xlabel('PT pseudospace', fontsize=8)
        if i == 0:
            ax.set_title('C · Specimen means', loc='left', fontsize=9)
        lfc = segment_lfc.reindex([gene]).iloc[0].reindex(list(segments))
        bar = fig.add_subplot(sub[i, 1])
        bar.bar(range(3), lfc.to_numpy(), color=[HUMAN if v > 0 else MOUSE for v in lfc.fillna(0)], width=.6)
        bar.axhline(0, color='#999999', lw=.6)
        bar.set_xticks(range(3), ['S1', 'S2', 'S3'] if i == len(drivers) - 1 else ['', '', ''], fontsize=7)
        bar.tick_params(labelsize=7)
        if i == 0:
            bar.set_title('D · Segment log2FC', loc='left', fontsize=9)
        records.append({'theme': theme, 'gene': gene, **{f'lfc_{s}': lfc.get(s) for s in segments},
                        **gene_table.loc[gene, ['T_spatial', 'shape_amplitude', 'peak_delta', 'peak_delta_position',
                                                'delta_mean', 'reverses_along_PT', 'detection_mouse',
                                                'detection_human']].to_dict()})
    ax = fig.add_subplot(gs[1, 0])
    ax.axis('off')
    ax.legend(*handles, frameon=False, fontsize=7, loc='upper left', bbox_to_anchor=(0, 1.02))
    counts_text = theme_rows.pseudospace_class.value_counts()
    ax.text(0, .38, f'{len(theme_rows)} confident pathways\n'
            + '\n'.join(f'{v} {k}' for k, v in counts_text.items()), va='top', fontsize=7.5)
    fig.suptitle(theme, x=0.01, y=.98, ha='left', fontweight='bold')
    save(fig, 'fig4_theme_' + re.sub(r'[^a-z]+', '_', theme.lower()).strip('_'))
    plt.show()
    plt.close(fig)
    return records


driver_records = []
for theme in theme_counts.sort_values('pathways', ascending=False).index:
    if theme_counts.loc[theme, 'pathways'] >= 3 and theme != 'Other / signaling':
        driver_records.extend(theme_figure(theme))
pd.DataFrame(driver_records).to_csv(output / 'theme_driver_genes.csv', index=False)

## 14 · Export and reproducibility record

In [ ]:
export_columns = ['pathway_id', 'library', 'name', 'theme', 'n_tested', 'effect_T_spatial', 'q_spatial', 'q_level',
                  'peak_position', 'median_member_z_at_peak', 'affected_width', 'retention_fraction', 'q_corr',
                  'candidate', 'specimen_sensitive', 'stable', 'correlation_supported', 'confident',
                  'conventional_species_call', 'conventional_call_also_relabeled', 'pseudospace_class',
                  'leading_edge_genes', 'spatial_driver_genes']
table = final[export_columns].merge(confident[['pathway_id', 'reversing_members', 'reversing_fraction']],
                                    on='pathway_id', how='left')
table.sort_values(['confident', 'q_spatial'], ascending=[False, True]).to_csv(output / 'final_pathway_table.csv', index=False)
manifest = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'alpha': ALPHA, 'n_null': N_NULL, 'seed': SEED,
            'basis_df': BASIS_DF, 'n_bins': N_BINS, 'min_retention': MIN_RETENTION,
            'partitions': {k: list(v) for k, v in partitions.items()},
            'n_structures': int(len(position)), 'n_genes': len(genes), 'n_pathways': len(gene_sets),
            'n_count_universe_genes': int(len(count_genes)), 'n_count_universe_pathways': len(count_sets),
            'funnel': funnel.to_dict(), 'class_counts': class_counts.to_dict(),
            'specificity': specificity[['true_hits', 'null_mean', 'ncdr']].round(4).to_dict(orient='index'),
            'caveat': 'Descriptive: two mice and two cortex sections from one male human donor; two relabelings '
                      'screen for gross nonspecificity and are not a calibrated false-discovery rate.'}
(output / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
print(json.dumps({k: manifest[k] for k in ('funnel', 'class_counts')}, indent=2))